# 06 — CNN feature extractor

Train on train windows; saliency / eval preds on **selection** only.


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

from spy_alpha_research.pipelines import fit_cnn
from spy_alpha_research.reporting import plot_importance

X_train = load_frame(PROC / "features_train_scaled.parquet")
X_sel = load_frame(PROC / "features_selection_scaled.parquet")
y = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
cols = [c for c in ["ret_lag_1", "vol_21", "volume_z_21"] if c in X_train.columns]
panel_tr = X_train[cols] if cols else X_train.iloc[:, :3]
panel_sel = X_sel[cols] if cols else X_sel.iloc[:, :3]
cnn_cfg = {k: v for k, v in models["cnn"].items() if k != "channels"}
result = fit_cnn(
    panel_tr, y.reindex(panel_tr.index),
    panel_sel, y.reindex(panel_sel.index),
    seed=exp["project"]["seed"], **cnn_cfg,
)
plot_importance(result.channel_saliency, FIG / "cnn_saliency.png", title="CNN saliency (selection)")
result.channel_saliency
